# Ω-Point Architect: Qwen2.5-0.5B TPU Trainer (V1.1)

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** on the **Architect** synthetic thinking logs. 

### Hardware: Kaggle TPU v3-8 / v5e
### Precision: BFloat16
### Method: Full Parameter Fine-Tuning

**Note:** Ensure you have uploaded `prepared_chatml_dataset.jsonl` as a Kaggle Dataset and attached it to this notebook.

In [ ]:
!pip install "transformers>=4.40.0" "datasets>=2.18.0" "trl>=0.8.6" "accelerate>=0.29.0" "peft>=0.10.0"

In [ ]:
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer,
)
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# TPU Optimization
os.environ['XLA_USE_BF16'] = '1'
os.environ['PJRT_DEVICE'] = 'TPU'

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_PATH = "/kaggle/input/your-dataset-name/prepared_chatml_dataset.jsonl" # UPDATE THIS
OUTPUT_DIR = "/kaggle/working/qwen2.5-architect-v1"

def train_fn(index, flags):
    torch.manual_seed(42)
    device = xm.xla_device()
    
    # 1. Load Tokenizer & Add Thinking Tags
    xm.master_print(f"Initializing tokenizer for {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    special_tokens = {
        "additional_special_tokens": ["<think-out>", "</think-out>", "<think-in>", "</think-in>"]
    }
    tokenizer.add_special_tokens(special_tokens)
    tokenizer.pad_token = tokenizer.eos_token
    
    # 2. Load Model & Resize for new tokens
    xm.master_print(f"Loading model: {MODEL_ID}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        torch_dtype=torch.bfloat16
    ).to(device)
    model.resize_token_embeddings(len(tokenizer))
    
    # 3. Load ChatML Dataset
    if not os.path.exists(DATASET_PATH):
        xm.master_print(f"CRITICAL ERROR: Dataset not found at {DATASET_PATH}")
        return

    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    
    # 4. Training Arguments (v5e-8 Optimized with SFTConfig)
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4, 
        gradient_accumulation_steps=4,
        learning_rate=1e-5, 
        num_train_epochs=3,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        bf16=True,
        report_to="none",
        ddp_find_unused_parameters=False,
        max_seq_length=1024,
        packing=False
    )
    
    # 5. Trainer Initialization
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        tokenizer=tokenizer,
        args=args
    )
    
    # 6. Execute Training
    xm.master_print("Commencing Architect Fine-Tuning...")
    trainer.train()
    
    # 7. Save Final Model (Master process only)
    xm.master_print("Training Complete. Saving model...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    xm.master_print(f"Model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    flags = {}
    xmp.spawn(train_fn, args=(flags,), nprocs=8, start_method='fork')

## Post-Training Inference Check

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

prompt = "<|im_start|>system\nYou are the ARCHITECT of the Brain Vat.<|im_end|>\n<|im_start|>user\n[MAUK]: mud mud mud...<|im_end|>\n<|im_start|>assistant\n<think-out>"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))